# Backpropagation as Reverse-Mode Automatic Differentiation

**Goal:** understand backprop as one instance of a general algorithm, *reverse-mode automatic differentiation*, then build a small autograd engine in NumPy and verify it.

By the end you should be able to:

 - explain why reverse mode computes a full gradient in one backward pass, and when forward mode is the better choice
 - derive the vector–Jacobian product (VJP) of any operation with matrix differentials, without ever forming a Jacobian
 - explain why gradients *accumulate* (fan-out) and why nodes are processed in reverse topological order
 - derive $\boldsymbol{\delta}^{(L)} = \hat{\mathbf{y}} - \mathbf{y}$ for softmax + cross-entropy
 - verify any gradient numerically, and know what error to expect

## 1. Programs as compositions

A differentiable program is a composition

$$f = f_L \circ f_{L-1} \circ \cdots \circ f_1 : \mathbb{R}^{d_0} \to \mathbb{R}^{d_L}, \qquad \mathbf{x}_\ell = f_\ell(\mathbf{x}_{\ell-1}).$$

Let $\mathbf{J}_\ell = \dfrac{\partial \mathbf{x}_\ell}{\partial \mathbf{x}_{\ell-1}} \in \mathbb{R}^{d_\ell \times d_{\ell-1}}$ be the Jacobian of step $\ell$, evaluated at $\mathbf{x}_{\ell-1}$. The chain rule says

$$\mathbf{J}_f(\mathbf{x}_0) = \mathbf{J}_L \, \mathbf{J}_{L-1} \cdots \mathbf{J}_1 .$$

In training, $f$ is the **loss as a function of the parameters**: $d_0 = P$ (millions to trillions) and $d_L = 1$. So $\mathbf{J}_f$ is a single row, $\nabla\mathcal{L}^\top$.

## 2. Two ways to evaluate the same product

Matrix multiplication is associative, so the product can be bracketed from either end. The answer is the same; the **cost** is not.

**Forward mode** (right to left) pushes a *tangent* $\mathbf{v} \in \mathbb{R}^{d_0}$ through the program:

$$\mathbf{J}_f\mathbf{v} = \mathbf{J}_L\big(\cdots\big(\mathbf{J}_2(\mathbf{J}_1\mathbf{v})\big)\big), \qquad \dot{\mathbf{x}}_\ell = \mathbf{J}_\ell\,\dot{\mathbf{x}}_{\ell-1},\quad \dot{\mathbf{x}}_0 = \mathbf{v}.$$

One pass gives one Jacobian–vector product (JVP), a directional derivative. To get the full gradient you seed $\mathbf{v} = \mathbf{e}_1, \dots, \mathbf{e}_{d_0}$: **$d_0$ passes**.

**Reverse mode** (left to right) pulls a *cotangent* $\mathbf{u} \in \mathbb{R}^{d_L}$ backwards:

$$\mathbf{u}^\top\mathbf{J}_f = \big(\big((\mathbf{u}^\top\mathbf{J}_L)\mathbf{J}_{L-1}\big)\cdots\big)\mathbf{J}_1, \qquad \mathbf{g}_{\ell-1} = \mathbf{J}_\ell^\top\,\mathbf{g}_\ell,\quad \mathbf{g}_L = \mathbf{u}.$$

One pass gives one vector–Jacobian product (VJP), a full row of $\mathbf{J}_f$. With a scalar loss, $\mathbf{u} = 1$ and **one pass gives the whole gradient**. Here $\mathbf{g}_\ell = \partial\mathcal{L}/\partial\mathbf{x}_\ell$ is "the gradient at node $\ell$".

| | propagates | one pass gives | passes for all of $\mathbf{J}_f$ | extra memory |
|---|---|---|---|---|
| Forward | tangent $\dot{\mathbf{x}}$ | $\mathbf{J}_f\mathbf{v}$ (a column) | $d_0$ (number of inputs) | $O(1)$: intermediates can be discarded |
| Reverse | cotangent $\mathbf{g}$ | $\mathbf{u}^\top\mathbf{J}_f$ (a row) | $d_L$ (number of outputs) | all intermediates (the *tape*) |

For a neural network, $d_0 = P$ and $d_L = 1$, so reverse mode is cheaper by a factor of $P$.

Two more points:

 - **Only vector–matrix products, never matrix–matrix.** Multiplying the Jacobians together first would cost $O(d^3)$ per step. Pushing a vector through costs $O(d^2)$, and $O(d)$ for elementwise steps, whose Jacobian is diagonal and never materialized.
 - **The cheap gradient principle.** The cost of $\nabla f$ is at most a small constant times the cost of $f$, *independent of $P$*. Griewank & Walther bound the constant by about 4–5 in their operation-count model; for a dense layer the backward pass is two matmuls against one forward, so about 2×. Compare finite differences: $2P$ evaluations of $f$ (central), plus truncation and roundoff error. That is a factor-of-$P$ saving, not an exponential one.

## 3. General graphs: fan-out means summing

Real programs are DAGs, not chains. If node $v$ is consumed by children $c_1, \dots, c_k$, the multivariable chain rule gives

$$\mathbf{g}_v = \sum_{i=1}^{k} \left(\frac{\partial c_i}{\partial v}\right)^{\!\top} \mathbf{g}_{c_i}.$$

Two consequences for the implementation:

1. **Accumulate** gradients with `+=`. Never overwrite.
2. Process nodes in **reverse topological order**, so $\mathbf{g}_v$ has received every contribution before it is pushed on to $v$'s parents.

*Example:* $y = x \cdot x$. Both inputs of the product are the same node, so $x$ receives two contributions $x\,g_y$, giving $2x\,g_y$ as it should.

*Parameter sharing is fan-out.* The recurrent weight $\mathbf{W}_h$ used at every time step, or a conv kernel used at every position, is one node with many children. The $\sum_t$ in backpropagation through time is exactly this sum.

## 4. Deriving VJPs without forming Jacobians

Use the Frobenius inner product $\langle \mathbf{A}, \mathbf{B}\rangle = \operatorname{tr}(\mathbf{A}^\top\mathbf{B}) = \sum_{ij} A_{ij}B_{ij}$. For a scalar loss depending on a node $\mathbf{C}$, define $\mathbf{G}_\mathbf{C} = \partial\mathcal{L}/\partial\mathbf{C}$ (same shape as $\mathbf{C}$), so that

$$d\mathcal{L} = \langle \mathbf{G}_\mathbf{C},\, d\mathbf{C}\rangle.$$

**Recipe:** write $d\mathbf{C}$ in terms of the input differentials, then use cyclicity of the trace to rearrange each term into the form $\langle\,\cdot\,, d\mathbf{A}\rangle$. Whatever sits in the first slot is $\mathbf{G}_\mathbf{A}$.

**Matmul.** $\mathbf{C} = \mathbf{A}\mathbf{B}$ with $\mathbf{A} \in \mathbb{R}^{m\times k}$, $\mathbf{B} \in \mathbb{R}^{k\times n}$. Then $d\mathbf{C} = d\mathbf{A}\,\mathbf{B} + \mathbf{A}\,d\mathbf{B}$, and

$$\langle \mathbf{G}, d\mathbf{A}\,\mathbf{B}\rangle = \operatorname{tr}(\mathbf{G}^\top d\mathbf{A}\,\mathbf{B}) = \operatorname{tr}(\mathbf{B}\mathbf{G}^\top d\mathbf{A}) = \langle \mathbf{G}\mathbf{B}^\top, d\mathbf{A}\rangle, \qquad \langle \mathbf{G}, \mathbf{A}\,d\mathbf{B}\rangle = \langle \mathbf{A}^\top\mathbf{G}, d\mathbf{B}\rangle.$$

$$\boxed{\mathbf{G}_\mathbf{A} = \mathbf{G}\mathbf{B}^\top \in \mathbb{R}^{m\times k}, \qquad \mathbf{G}_\mathbf{B} = \mathbf{A}^\top\mathbf{G} \in \mathbb{R}^{k\times n}}$$

Shapes have to match, which is a free sanity check.

**Elementwise.** $\mathbf{Y} = \phi(\mathbf{X})$ gives $d\mathbf{Y} = \phi'(\mathbf{X}) \odot d\mathbf{X}$, so $\langle\mathbf{G}, \phi'(\mathbf{X})\odot d\mathbf{X}\rangle = \langle \mathbf{G}\odot\phi'(\mathbf{X}), d\mathbf{X}\rangle$ and $\mathbf{G}_\mathbf{X} = \mathbf{G}\odot\phi'(\mathbf{X})$.

**Broadcasting and summation are adjoint.** A bias is broadcast over the $N$ rows of a batch: $\mathbf{Y} = \mathbf{X}\mathbf{W} + \mathbf{1}\mathbf{b}^\top$. Then $d\mathbf{Y} = \mathbf{1}\,d\mathbf{b}^\top$ and

$$\langle \mathbf{G}, \mathbf{1}\,d\mathbf{b}^\top\rangle = \operatorname{tr}(\mathbf{G}^\top\mathbf{1}\,d\mathbf{b}^\top) = d\mathbf{b}^\top\mathbf{G}^\top\mathbf{1} = \langle \mathbf{G}^\top\mathbf{1}, d\mathbf{b}\rangle,$$

so $\mathbf{g}_\mathbf{b} = \mathbf{G}^\top\mathbf{1}$: the column sums of $\mathbf{G}$, i.e. **the bias gradient is $\boldsymbol{\delta}$ summed over the batch**. In the other direction, the VJP of a sum is a broadcast.

**Log-softmax** (one row $\mathbf{x} \in \mathbb{R}^K$). $\mathbf{y} = \mathbf{x} - \operatorname{lse}(\mathbf{x})\mathbf{1}$ with $\operatorname{lse}(\mathbf{x}) = \log\sum_j e^{x_j}$, whose gradient is the softmax $\mathbf{s}$. So $d\mathbf{y} = d\mathbf{x} - (\mathbf{s}^\top d\mathbf{x})\mathbf{1}$ and

$$\langle \mathbf{g}, d\mathbf{y}\rangle = \mathbf{g}^\top d\mathbf{x} - (\mathbf{1}^\top\mathbf{g})(\mathbf{s}^\top d\mathbf{x}) \quad\Longrightarrow\quad \mathbf{g}_\mathbf{x} = \mathbf{g} - (\mathbf{1}^\top\mathbf{g})\,\mathbf{s}.$$

**Softmax + cross-entropy.** With a one-hot label $\mathbf{y}$, $\mathcal{L} = -\mathbf{y}^\top\operatorname{logsoftmax}(\mathbf{x})$. The gradient arriving at the log-softmax output is $-\mathbf{y}$, so

$$\mathbf{g}_\mathbf{x} = -\mathbf{y} - \big(\mathbf{1}^\top(-\mathbf{y})\big)\mathbf{s} = \mathbf{s} - \mathbf{y} \qquad (\text{since } \mathbf{1}^\top\mathbf{y} = 1).$$

For a batch mean, $\mathbf{G}_\mathbf{X} = (\mathbf{S} - \mathbf{Y})/N$. This is the correct $\boldsymbol{\delta}^{(L)}$ for a softmax output. It is *not* $\nabla_{\mathbf{h}}\mathcal{L}\circ\phi'(\mathbf{z})$, because the softmax Jacobian is not diagonal; but the two nonlinearities fuse into something very simple. Frameworks fuse them in code too, which is also more numerically stable.

## 5. Reading the backprop notes as reverse mode

In the column convention of `NeuralNetworksNotes.ipynb` ($\mathbf{z}^{(\ell)} = \mathbf{W}^{(\ell)}\bar{\mathbf{h}}^{(\ell-1)}$):

 - $\boldsymbol{\delta}^{(\ell)}$ is just $\mathbf{g}_{\mathbf{z}^{(\ell)}}$, the cotangent at the pre-activation.
 - The recursion is two VJPs in a row: matmul gives $\mathbf{g}_{\mathbf{h}^{(\ell)}} = (\mathbf{W}^{(\ell+1)})^\top\boldsymbol{\delta}^{(\ell+1)}$, then elementwise gives $\boldsymbol{\delta}^{(\ell)} = \mathbf{g}_{\mathbf{h}^{(\ell)}}\odot\phi'(\mathbf{z}^{(\ell)})$.
 - $\nabla_{\mathbf{W}^{(\ell)}}\mathcal{L} = \boldsymbol{\delta}^{(\ell)}(\bar{\mathbf{h}}^{(\ell-1)})^\top$ is matmul's *other* VJP.

**The bias fix.** With an augmented matrix $\tilde{\mathbf{W}}^{(\ell+1)} = [\,\mathbf{W} \mid \mathbf{b}\,]$, the matmul VJP returns a gradient for the augmented vector $\bar{\mathbf{h}}^{(\ell)}$, which has $d_\ell + 1$ entries. The last entry is the gradient with respect to the constant 1 and is thrown away:

$$\mathbf{g}_{\mathbf{h}^{(\ell)}} = \big(\tilde{\mathbf{W}}^{(\ell+1)}_{:,\,1:d_\ell}\big)^\top\boldsymbol{\delta}^{(\ell+1)}.$$

Keeping $\mathbf{b}$ as a separate node, as the code below does, avoids the issue entirely.

**Convention used below:** rows are examples, so a layer is $\mathbf{Z} = \mathbf{H}\mathbf{W} + \mathbf{b}$ with $\mathbf{H} \in \mathbb{R}^{N\times d_\text{in}}$ and $\mathbf{W} \in \mathbb{R}^{d_\text{in}\times d_\text{out}}$. Everything transposes: $\mathbf{G}_\mathbf{W} = \mathbf{H}^\top\mathbf{G}_\mathbf{Z}$, $\mathbf{G}_\mathbf{H} = \mathbf{G}_\mathbf{Z}\mathbf{W}^\top$, and $\mathbf{g}_\mathbf{b}$ = column sums of $\mathbf{G}_\mathbf{Z}$.

## 6. The price of reverse mode: memory

The backward pass needs values from the forward pass: $\mathbf{G}_\mathbf{W} = \mathbf{H}^\top\mathbf{G}_\mathbf{Z}$ needs $\mathbf{H}$, and the tanh VJP needs $1 - \tanh^2$. So the tape keeps every intermediate. For an MLP that is $O(L \cdot N \cdot d)$ memory, and in practice **activations, not parameters, usually dominate training memory**.

**Gradient checkpointing** trades compute for memory: store only every $\sqrt{L}$-th activation and recompute each segment during the backward pass. Memory drops to $O(\sqrt{L})$ at the cost of roughly one extra forward pass.

Forward mode has no such cost. That is one reason it is used for JVPs and for Hessian–vector products ("forward-over-reverse").

## 7. Implementation

Each `Tensor` stores its value, its parents, and a **VJP closure** that maps the gradient at this node to the gradient contributions for its parents. `backward()` does exactly what sections 2–3 say:

1. topologically sort the graph,
2. seed the output with $\mathbf{g} = 1$,
3. walk in reverse order, pushing each node's gradient to its parents with `+=`.

Design choices: everything is `float64` so gradient checks are meaningful; every node gets a gradient (no `requires_grad`); `@` supports 2-D arrays only.

In [1]:
import numpy as np


def _unbroadcast(g, shape):
    """VJP of NumPy broadcasting: sum g back down to `shape` (section 4: broadcast and sum are adjoint)."""
    while g.ndim > len(shape):
        g = g.sum(axis=0)
    for i, n in enumerate(shape):
        if n == 1 and g.shape[i] != 1:
            g = g.sum(axis=i, keepdims=True)
    return g


def _lift(x):
    return x if isinstance(x, Tensor) else Tensor(x)


def _node(data, parents, vjp, op):
    out = Tensor(data, parents, op)
    out._vjp = vjp
    return out


def _topo(root):
    """Iterative post-order DFS: every node appears after all of its parents."""
    order, seen, stack = [], set(), [(root, False)]
    while stack:
        node, expanded = stack.pop()
        if expanded:
            order.append(node)
            continue
        if id(node) in seen:
            continue
        seen.add(id(node))
        stack.append((node, True))
        stack.extend((p, False) for p in node._parents if id(p) not in seen)
    return order


class Tensor:
    """A node in the computation graph: a value plus a rule for pulling gradients back to its parents."""

    def __init__(self, data, parents=(), op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.grad = None
        self._parents = parents
        self._vjp = lambda g: ()
        self.op = op

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op!r})"

    def backward(self):
        """Reverse-mode sweep: fills .grad on every node in this scalar's graph."""
        assert self.data.size == 1, "backward() needs a scalar output"
        order = _topo(self)
        for t in order:
            t.grad = np.zeros_like(t.data)
        self.grad = np.ones_like(self.data)
        for t in reversed(order):
            for parent, g in zip(t._parents, t._vjp(t.grad)):
                parent.grad += g  # fan-out: contributions accumulate

    # --- arithmetic (broadcasting allowed) ---
    def __add__(self, other):
        a, b = self, _lift(other)
        return _node(a.data + b.data, (a, b),
                     lambda g: (_unbroadcast(g, a.shape), _unbroadcast(g, b.shape)), "+")

    def __mul__(self, other):
        a, b = self, _lift(other)
        return _node(a.data * b.data, (a, b),
                     lambda g: (_unbroadcast(g * b.data, a.shape), _unbroadcast(g * a.data, b.shape)), "*")

    def __pow__(self, p):
        a = self  # p is a Python number
        return _node(a.data ** p, (a,), lambda g: (g * p * a.data ** (p - 1),), f"**{p}")

    def __matmul__(self, other):
        a, b = self, _lift(other)
        assert a.data.ndim == 2 and b.data.ndim == 2, "2-D matmul only"
        return _node(a.data @ b.data, (a, b), lambda g: (g @ b.data.T, a.data.T @ g), "@")

    __radd__ = __add__
    __rmul__ = __mul__

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-_lift(other))

    def __rsub__(self, other):
        return _lift(other) + (-self)

    def __truediv__(self, other):
        return self * _lift(other) ** -1.0

    def __rtruediv__(self, other):
        return _lift(other) * self ** -1.0

    # --- reductions ---
    def sum(self, axis=None, keepdims=False):
        a = self

        def vjp(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, a.shape),)  # VJP of a sum is a broadcast

        return _node(a.data.sum(axis=axis, keepdims=keepdims), (a,), vjp, "sum")

    def mean(self, axis=None, keepdims=False):
        out = self.sum(axis=axis, keepdims=keepdims)
        return out * (out.data.size / self.data.size)

    # --- elementwise nonlinearities: G_X = G ⊙ φ'(X) ---
    def exp(self):
        out = np.exp(self.data)
        return _node(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        a = self
        return _node(np.log(a.data), (a,), lambda g: (g / a.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return _node(out, (self,), lambda g: (g * (1.0 - out ** 2),), "tanh")

    def relu(self):
        a = self
        return _node(np.maximum(a.data, 0.0), (a,), lambda g: (g * (a.data > 0),), "relu")

    def log_softmax(self, axis=-1):
        x = self.data
        m = x.max(axis=axis, keepdims=True)
        out = x - (m + np.log(np.exp(x - m).sum(axis=axis, keepdims=True)))  # stable log-sum-exp
        s = np.exp(out)
        # section 4: g_x = g - (1ᵀg) s
        return _node(out, (self,), lambda g: (g - s * g.sum(axis=axis, keepdims=True),), "log_softmax")


def cross_entropy(logits, Y):
    """Mean cross-entropy; Y is one-hot with shape (N, K)."""
    N = logits.shape[0]
    return -(logits.log_softmax(axis=1) * Y).sum() / N

In [2]:
# Fan-out sanity check: y = x*x + x  =>  dy/dx = 2x + 1
x = Tensor(3.0)
y = x * x + x
y.backward()
print("dy/dx at x=3:", x.grad, "(expected 7.0)")

dy/dx at x=3: 7.0 (expected 7.0)


## 8. Forward mode in a few lines: dual numbers

Forward mode can be implemented with **dual numbers** $a + b\varepsilon$, where $\varepsilon^2 = 0$. The Taylor series terminates after one term,

$$f(a + b\varepsilon) = f(a) + f'(a)\,b\,\varepsilon,$$

so carrying the $\varepsilon$-part through every operation propagates a tangent. To get $\partial f/\partial x_i$ you seed $x_i$ with tangent 1 and the rest with 0, which takes **one pass per input**. Reverse mode gets all of them in one pass.

In [3]:
class Dual:
    """a + bε with ε² = 0: `val` is the value, `tan` the tangent."""

    def __init__(self, val, tan=0.0):
        self.val, self.tan = val, tan

    def __add__(self, o):
        return Dual(self.val + o.val, self.tan + o.tan)

    def __mul__(self, o):
        return Dual(self.val * o.val, self.val * o.tan + self.tan * o.val)  # product rule


def dual_tanh(x):
    t = np.tanh(x.val)
    return Dual(t, (1.0 - t ** 2) * x.tan)


def f(x1, x2, x3, tanh):
    return tanh(x1 * x2) + x2 * x3 * x3


point = [0.5, -1.2, 2.0]

# forward mode: one pass per input
fwd = []
for i in range(3):
    args = [Dual(v, 1.0 if j == i else 0.0) for j, v in enumerate(point)]
    fwd.append(f(*args, dual_tanh).tan)

# reverse mode: one pass for all inputs
ts = [Tensor(v) for v in point]
f(*ts, lambda t: t.tanh()).backward()
rev = [float(t.grad) for t in ts]

print("forward mode (3 passes):", np.round(fwd, 10))
print("reverse mode (1 pass):  ", np.round(rev, 10))

forward mode (3 passes): [-0.85389332  4.35578888 -4.8       ]
reverse mode (1 pass):   [-0.85389332  4.35578888 -4.8       ]


## 9. Gradient checking

The central difference has error from two sources:

$$\frac{f(\mathbf{x}+h\mathbf{e}_i) - f(\mathbf{x}-h\mathbf{e}_i)}{2h} = \partial_i f(\mathbf{x}) + \underbrace{O(h^2)}_{\text{truncation}} + \underbrace{O(\varepsilon_\text{mach}/h)}_{\text{roundoff}}.$$

Balancing the two gives $h \sim \varepsilon_\text{mach}^{1/3} \approx 6\times10^{-6}$ in float64, with error around $\varepsilon_\text{mach}^{2/3} \approx 10^{-11}$. So relative errors of $10^{-8}$ to $10^{-11}$ mean the gradient is right. Anything above about $10^{-5}$ is a bug.

It costs $2P$ evaluations, which is why it is used for testing only. It also breaks at kinks (ReLU at 0), so random inputs are used, and they almost surely avoid hitting a kink exactly.

The relative error used is $\|\mathbf{g}_\text{auto} - \mathbf{g}_\text{num}\| \,/\, (\|\mathbf{g}_\text{auto}\| + \|\mathbf{g}_\text{num}\|)$.

In [4]:
def numerical_grad(f, x, h=1e-6):
    """Central differences of the scalar function f() with respect to the array x, perturbed in place."""
    g = np.zeros_like(x)
    for i in np.ndindex(x.shape):
        old = x[i]
        x[i] = old + h
        f_plus = f()
        x[i] = old - h
        f_minus = f()
        x[i] = old
        g[i] = (f_plus - f_minus) / (2 * h)
    return g


def grad_check(name, build, *arrays):
    """build(*tensors) -> scalar Tensor. Compares backward() with central differences for every input."""
    leaves = [Tensor(a) for a in arrays]
    build(*leaves).backward()
    worst = 0.0
    for leaf in leaves:
        num = numerical_grad(lambda: build(*[Tensor(l.data) for l in leaves]).data.item(), leaf.data)
        err = np.linalg.norm(leaf.grad - num) / (np.linalg.norm(leaf.grad) + np.linalg.norm(num) + 1e-300)
        worst = max(worst, err)
    print(f"{name:<34} max relative error {worst:.2e}  {'ok' if worst < 1e-6 else 'FAIL'}")
    return worst


rng = np.random.default_rng(0)
A, B, R = rng.normal(size=(4, 3)), rng.normal(size=(4, 3)), rng.normal(size=(4, 3))
W, c = rng.normal(size=(3, 5)), rng.normal(size=(4, 5))
b_row, b_vec, col = rng.normal(size=(1, 3)), rng.normal(size=(3,)), rng.normal(size=(4, 1))

checks = [
    ("add, broadcast (1,3)",       lambda a, b: ((a + b).tanh() * R).sum(),          A, b_row),
    ("add, broadcast (3,)",        lambda a, b: ((a + b).tanh() * R).sum(),          A, b_vec),
    ("mul, broadcast (4,1)",       lambda a, b: ((a * b).tanh() * R).sum(),          A, col),
    ("div and pow",                lambda a, b: (a / (b ** 2 + 1.0) * R).sum(),      A, B),
    ("matmul",                     lambda a, w: ((a @ w).tanh() * c).sum(),          A, W),
    ("exp and log",                lambda a: ((a.exp() + 1.0).log() * R).mean(),     A),
    ("relu",                       lambda a, b: (a.relu() * b).sum(),                A, B),
    ("sum over an axis",           lambda a: (a.sum(axis=0) ** 2).sum(),             A),
    ("mean, keepdims",             lambda a: ((a.mean(axis=1, keepdims=True) * a).tanh()).sum(), A),
    ("log_softmax",                lambda a: (a.log_softmax(axis=1) * R).sum(),      A),
    ("fan-out (a used 3 times)",   lambda a: ((a * a).tanh() * a * R).sum(),         A),
]
results = [grad_check(*chk) for chk in checks]

add, broadcast (1,3)               max relative error 1.88e-10  ok
add, broadcast (3,)                max relative error 1.81e-10  ok
mul, broadcast (4,1)               max relative error 8.70e-11  ok
div and pow                        max relative error 1.72e-10  ok
matmul                             max relative error 7.61e-11  ok
exp and log                        max relative error 1.99e-10  ok
relu                               max relative error 4.77e-11  ok
sum over an axis                   max relative error 3.82e-11  ok
mean, keepdims                     max relative error 8.79e-11  ok
log_softmax                        max relative error 1.86e-10  ok
fan-out (a used 3 times)           max relative error 6.78e-11  ok


In [5]:
# Section 4's result: for softmax + cross-entropy, the gradient at the logits is (S - Y)/N
N, K = 6, 4
logits = Tensor(rng.normal(size=(N, K)))
Y = np.eye(K)[rng.integers(0, K, size=N)]
cross_entropy(logits, Y).backward()

S = np.exp(logits.data) / np.exp(logits.data).sum(axis=1, keepdims=True)
print("max |autograd - (S - Y)/N| =", np.abs(logits.grad - (S - Y) / N).max())

max |autograd - (S - Y)/N| = 2.7755575615628914e-17


## 10. Training an MLP with the engine

A 2-D, 3-class spiral dataset: classes interleave, so no linear model can separate them. The MLP is 2 → 64 → 64 → 3 with ReLU and **He initialization**, $\mathbf{W}\sim\mathcal{N}(0, 2/d_\text{in})$ (module 3 derives where the 2 comes from). Training is full-batch gradient descent on mean cross-entropy.

The whole network is first gradient-checked end to end.

In [6]:
def spirals(n_per_class, n_classes, rng, noise=0.2):
    X, y = [], []
    for k in range(n_classes):
        r = np.linspace(0.0, 1.0, n_per_class)
        t = np.linspace(4.0 * k, 4.0 * (k + 1), n_per_class) + rng.normal(0.0, noise, n_per_class)
        X.append(np.c_[r * np.sin(t), r * np.cos(t)])
        y.append(np.full(n_per_class, k))
    return np.concatenate(X), np.concatenate(y)


class MLP:
    def __init__(self, sizes, rng):
        self.params = []
        for d_in, d_out in zip(sizes[:-1], sizes[1:]):
            self.params.append(Tensor(rng.normal(0.0, np.sqrt(2.0 / d_in), (d_in, d_out))))  # He init
            self.params.append(Tensor(np.zeros((1, d_out))))  # bias: a separate node, broadcast over the batch

    def __call__(self, X, params=None):
        params = self.params if params is None else params
        layers = list(zip(params[0::2], params[1::2]))
        h = _lift(X)
        for i, (W, b) in enumerate(layers):
            h = h @ W + b
            if i < len(layers) - 1:
                h = h.relu()
        return h


rng = np.random.default_rng(1)
X, y = spirals(100, 3, rng)
Y = np.eye(3)[y]
model = MLP([2, 64, 64, 3], rng)

# end-to-end gradient check of the whole network on a small batch
small = MLP([2, 8, 8, 3], np.random.default_rng(2))
check_mlp = lambda *ps: cross_entropy(small(X[:20], ps), Y[:20])
grad_check("full MLP, zero biases", check_mlp, *[p.data for p in small.params])

for b in small.params[1::2]:
    b.data[:] = rng.normal(0.0, 0.1, b.shape)
grad_check("full MLP, random biases", check_mlp, *[p.data for p in small.params])

full MLP, zero biases              max relative error 2.03e-02  FAIL
full MLP, random biases            max relative error 4.42e-09  ok


**Why did the first check fail?** It is not a bug in the engine. `X[0]` is exactly the origin (the spiral starts at $r = 0$), and with zero biases every first-layer pre-activation there is exactly $0$, which is the ReLU kink. At the kink, $\mathrm{ReLU}$ is not differentiable. The engine returns one valid choice (a subgradient, using $\mathbb{1}[z > 0]$), while the central difference averages the two one-sided slopes and gets $\tfrac12$. With random biases nothing sits exactly on a kink, and the check passes.

This is the classic failure mode of gradient checking. Look for kinks before you go hunting for a bug.

In [7]:
lr, steps = 0.5, 3000
for step in range(steps + 1):
    loss = cross_entropy(model(X), Y)
    loss.backward()
    for p in model.params:
        p.data -= lr * p.grad
    if step % 500 == 0:
        acc = (model(X).data.argmax(axis=1) == y).mean()
        print(f"step {step:>4}  loss {loss.data.item():.4f}  accuracy {acc:.3f}")

step    0  loss 0.9926  accuracy 0.520
step  500  loss 0.0331  accuracy 0.993
step 1000  loss 0.0236  accuracy 0.993
step 1500  loss 0.0224  accuracy 0.993
step 2000  loss 0.0205  accuracy 0.993
step 2500  loss 0.0192  accuracy 0.993
step 3000  loss 0.0182  accuracy 0.993


In [ ]:
# Decision regions (needs matplotlib)
import matplotlib.pyplot as plt

xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 300), np.linspace(-1.2, 1.2, 300))
pred = model(np.c_[xx.ravel(), yy.ravel()]).data.argmax(axis=1).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5, 5))
ax.contourf(xx, yy, pred, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis")
ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="viridis", edgecolors="k", linewidths=0.3)
ax.set_title("MLP trained with the mini autograd")
ax.set_aspect("equal")
plt.show()

## 11. Exercises

1. **Transpose and reshape.** Add `.T` and `.reshape(shape)` to `Tensor`. Derive their VJPs with the differential recipe, then gradient-check them.
2. **LayerNorm.** Derive the VJP of $\mathbf{y} = (\mathbf{x}-\mu)/\sqrt{\sigma^2+\epsilon}$ (one row) with differentials, implement it as a single op, and check it against the version composed from primitive ops. Which is cheaper, and why?
3. **Max and subgradients.** Implement `max(axis)`. What should the VJP do when there is a tie? Is your choice a subgradient of a convex function?
4. **The dot-product test.** Add forward mode to `Tensor` (a `jvp` per op). Then verify $\langle\mathbf{u}, \mathbf{J}\mathbf{v}\rangle = \langle\mathbf{J}^\top\mathbf{u}, \mathbf{v}\rangle$ for random $\mathbf{u}, \mathbf{v}$ on the MLP. This checks JVPs against VJPs with no finite differences at all.
5. **Order matters.** Build a small graph where processing nodes in breadth-first order from the output (instead of reverse topological order) gives a wrong gradient. Explain which node is pushed on too early.
6. **Memory.** Count how many arrays the tape holds for an $L$-layer MLP. Then implement checkpointing for `MLP.__call__` that keeps only every $k$-th activation and recomputes the rest during `backward`.
7. **Hessian–vector products.** Using $\mathbf{H}\mathbf{v} = \nabla_\mathbf{x}\big(\nabla f(\mathbf{x})^\top\mathbf{v}\big)$, explain why an HVP costs about as much as a couple of gradients, while the full Hessian would cost $P$ of them.